In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

In [4]:
df = pd.read_csv("C:\\Users\\PREETHI HB\\Downloads\\optimized_learning_resource_recommendations_200.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Dataset shape: (1007, 16)


,user_id,course_id,timestamp,engagement_score,completion_status,predicted_engagement_score,assessment_score,feedback_score,age,education_level,learning_style,preferred_topics,title,topic,difficulty,content_type
0,U133,C015,2023-07-02,0.348821,In Progress,0.162183,64,4.40,49,Bachelor,Kinesthetic,Math,Course 15,Data Science,Beginner,Text
1,U029,C029,2023-06-02,0.110998,Completed,0.486442,81,4.43,45,Master,Visual,Art,Course 29,History,Advanced,Quiz
2,U132,C029,2023-11-11,0.808235,In Progress,0.738539,64,4.69,56,Bachelor,Visual,Music,Course 29,History,Advanced,Quiz
3,U138,C011,2023-11-30,0.947688,In Progress,0.866855,63,2.57,56,Master,Visual,AI,Course 11,Data Science,Advanced,Video
4,U184,C032,2023-05-04,0.072316,In Progress,0.626259,68,3.90,19,High School,Reading,Music,Course 32,Art,Beginner,Text


In [5]:
df = pd.read_csv(
    r"C:\Users\PREETHI HB\Downloads\optimized_learning_resource_recommendations_200.csv"
)

In [6]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDataset information:")
df.info()

Column names:
['user_id', 'course_id', 'timestamp', 'engagement_score', 'completion_status', 'predicted_engagement_score', 'assessment_score', 'feedback_score', 'age', 'education_level', 'learning_style', 'preferred_topics', 'title', 'topic', 'difficulty', 'content_type']

Missing values:
user_id                       0
course_id                     0
timestamp                     0
engagement_score              0
completion_status             0
predicted_engagement_score    0
assessment_score              0
feedback_score                0
age                           0
education_level               0
learning_style                0
preferred_topics              0
title                         0
topic                         0
difficulty                    0
content_type                  0
dtype: int64

Duplicate rows: 0

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 16 columns):
 #   Column                      Non-

In [7]:
required_columns = [
    "course_id",
    "title",
    "topic",
    "difficulty",
    "content_type",
    "learning_style",
    "engagement_score",
    "assessment_score"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing from dataset: {missing_columns}"
    )

# Remove rows missing essential course information
df = df.dropna(
    subset=[
        "course_id",
        "title",
        "topic",
        "difficulty",
        "content_type",
        "learning_style"
    ]
).copy()

# Convert text columns to clean strings
text_columns = [
    "course_id",
    "title",
    "topic",
    "difficulty",
    "content_type",
    "learning_style"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Remove rows with empty essential text
for col in text_columns:
    df = df[df[col] != ""]

# Keep one record per course for the recommendation catalog
courses = df.drop_duplicates(subset="course_id").reset_index(drop=True)

print("Cleaned interaction dataset:", df.shape)
print("Unique course catalog:", courses.shape)

display(courses.head())

Cleaned interaction dataset: (1007, 16)
Unique course catalog: (50, 16)


,user_id,course_id,timestamp,engagement_score,completion_status,predicted_engagement_score,assessment_score,feedback_score,age,education_level,learning_style,preferred_topics,title,topic,difficulty,content_type
0,U133,C015,2023-07-02,0.348821,In Progress,0.162183,64,4.40,49,Bachelor,Kinesthetic,Math,Course 15,Data Science,Beginner,Text
1,U029,C029,2023-06-02,0.110998,Completed,0.486442,81,4.43,45,Master,Visual,Art,Course 29,History,Advanced,Quiz
2,U138,C011,2023-11-30,0.947688,In Progress,0.866855,63,2.57,56,Master,Visual,AI,Course 11,Data Science,Advanced,Video
3,U184,C032,2023-05-04,0.072316,In Progress,0.626259,68,3.90,19,High School,Reading,Music,Course 32,Art,Beginner,Text
4,U145,C002,2023-04-24,0.955115,Not Started,0.669054,88,3.04,39,Bachelor,Visual,Math,Course 2,Math,Intermediate,Quiz


In [8]:
print("Available topics:")
print(sorted(courses["topic"].unique()))

print("\nAvailable difficulty levels:")
print(sorted(courses["difficulty"].unique()))

print("\nAvailable learning styles:")
print(sorted(courses["learning_style"].unique()))

print("\nAvailable content types:")
print(sorted(courses["content_type"].unique()))

Available topics:
['AI', 'Art', 'Data Science', 'History', 'Math', 'Music']

Available difficulty levels:
['Advanced', 'Beginner', 'Intermediate']

Available learning styles:
['Auditory', 'Kinesthetic', 'Reading', 'Visual']

Available content types:
['Quiz', 'Text', 'Video']


In [9]:
courses["features"] = (
    courses["title"].astype(str) + " " +
    courses["topic"].astype(str) + " " +
    courses["difficulty"].astype(str) + " " +
    courses["content_type"].astype(str) + " " +
    courses["learning_style"].astype(str)
)

print(courses[["course_id", "title", "features"]].head())

  course_id      title                                          features
0      C015  Course 15  Course 15 Data Science Beginner Text Kinesthetic
1      C029  Course 29            Course 29 History Advanced Quiz Visual
2      C011  Course 11      Course 11 Data Science Advanced Video Visual
3      C032  Course 32               Course 32 Art Beginner Text Reading
4      C002   Course 2            Course 2 Math Intermediate Quiz Visual


In [10]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

course_matrix = tfidf.fit_transform(courses["features"])

print("TF-IDF matrix shape:", course_matrix.shape)

TF-IDF matrix shape: (50, 59)


In [11]:
course_similarity = cosine_similarity(course_matrix)

print("Similarity matrix shape:", course_similarity.shape)

Similarity matrix shape: (50, 50)


In [12]:
def recommend_courses(
    topic,
    difficulty=None,
    learning_style=None,
    content_type=None,
    top_n=5
):
    # Start with all available courses
    results = courses.copy()

    # Filter by topic
    results = results[
        results["topic"].str.casefold() == topic.strip().casefold()
    ]

    if results.empty:
        print(f"No courses found for topic: {topic}")
        return pd.DataFrame()

    # Apply optional preferences
    if difficulty:
        matching = results[
            results["difficulty"].str.casefold()
            == difficulty.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    if learning_style:
        matching = results[
            results["learning_style"].str.casefold()
            == learning_style.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    if content_type:
        matching = results[
            results["content_type"].str.casefold()
            == content_type.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    # Build the student's preference description
    preference_text = " ".join(
        str(value) for value in [
            topic, difficulty or "",
            learning_style or "", content_type or ""
        ]
    )

    # Convert preferences using the trained TF-IDF vectorizer
    preference_vector = tfidf.transform([preference_text])

    # Calculate similarity against the selected courses
    selected_vectors = tfidf.transform(results["features"])

    scores = cosine_similarity(
        preference_vector, selected_vectors
    ).flatten()

    results = results.copy()
    results["Similarity Score (%)"] = (scores * 100).round(2)

    # Sort by relevance
    results = results.sort_values(
        by="Similarity Score (%)",
        ascending=False
    )

    output_columns = [
        "course_id",
        "title",
        "topic",
        "difficulty",
        "content_type",
        "learning_style",
        "Similarity Score (%)"
    ]

    return (
        results[output_columns]
        .head(top_n)
        .rename(columns={
            "course_id": "Course ID",
            "title": "Course Title",
            "topic": "Topic",
            "difficulty": "Difficulty",
            "content_type": "Content Type",
            "learning_style": "Learning Style"
        })
        .reset_index(drop=True)
    )

In [13]:
def recommend_courses(
    topic,
    difficulty=None,
    learning_style=None,
    content_type=None,
    top_n=5
):
    # Start with all available courses
    results = courses.copy()

    # Filter by topic
    results = results[
        results["topic"].str.casefold() == topic.strip().casefold()
    ]

    if results.empty:
        print(f"No courses found for topic: {topic}")
        return pd.DataFrame()

    # Apply optional preferences
    if difficulty:
        matching = results[
            results["difficulty"].str.casefold()
            == difficulty.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    if learning_style:
        matching = results[
            results["learning_style"].str.casefold()
            == learning_style.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    if content_type:
        matching = results[
            results["content_type"].str.casefold()
            == content_type.strip().casefold()
        ]
        if not matching.empty:
            results = matching

    # Build the student's preference description
    preference_text = " ".join(
        str(value) for value in [
            topic, difficulty or "",
            learning_style or "", content_type or ""
        ]
    )

    # Convert preferences using the trained TF-IDF vectorizer
    preference_vector = tfidf.transform([preference_text])

    # Calculate similarity against the selected courses
    selected_vectors = tfidf.transform(results["features"])

    scores = cosine_similarity(
        preference_vector, selected_vectors
    ).flatten()

    results = results.copy()
    results["Similarity Score (%)"] = (scores * 100).round(2)

    # Sort by relevance
    results = results.sort_values(
        by="Similarity Score (%)",
        ascending=False
    )

    output_columns = [
        "course_id",
        "title",
        "topic",
        "difficulty",
        "content_type",
        "learning_style",
        "Similarity Score (%)"
    ]

    return (
        results[output_columns]
        .head(top_n)
        .rename(columns={
            "course_id": "Course ID",
            "title": "Course Title",
            "topic": "Topic",
            "difficulty": "Difficulty",
            "content_type": "Content Type",
            "learning_style": "Learning Style"
        })
        .reset_index(drop=True)
    )

In [14]:
recommendations = recommend_courses(
    topic="Data Science",
    difficulty="Beginner",
    learning_style="Visual",
    content_type="Video",
    top_n=5
)

display(recommendations)

,Course ID,Course Title,Topic,Difficulty,Content Type,Learning Style,Similarity Score (%)
0,C038,Course 38,Data Science,Beginner,Quiz,Visual,60.43


In [15]:
ml_data = df.copy()

ml_data["engagement_score"] = pd.to_numeric(
    ml_data["engagement_score"], errors="coerce"
)

ml_data["assessment_score"] = pd.to_numeric(
    ml_data["assessment_score"], errors="coerce"
)

ml_data = ml_data.dropna(
    subset=["engagement_score", "assessment_score",
            "topic", "difficulty"]
)

X = ml_data[
    ["topic", "difficulty", "assessment_score"]
]

y = ml_data["engagement_score"]

print("Feature data shape:", X.shape)
print("Target data shape:", y.shape)

Feature data shape: (1007, 3)
Target data shape: (1007,)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 805
Testing rows: 202


In [17]:
categorical_features = ["topic", "difficulty"]
numeric_features = ["assessment_score"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        ("numeric", "passthrough", numeric_features)
    ]
)

engagement_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

engagement_model.fit(X_train, y_train)

print("Engagement model trained successfully!")

Engagement model trained successfully!


In [18]:
y_pred = engagement_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Compare with a simple average-based baseline
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print("Random Forest MSE:", round(mse, 4))
print("Random Forest R2 score:", round(r2, 4))

print("\nBaseline MSE:", round(baseline_mse, 4))
print("Baseline R2 score:", round(baseline_r2, 4))

Random Forest MSE: 0.1132
Random Forest R2 score: -0.3739

Baseline MSE: 0.0825
Baseline R2 score: -0.001


In [19]:
sample_student = pd.DataFrame({
    "topic": ["Data Science"],
    "difficulty": ["Beginner"],
    "assessment_score": [75]
})

predicted_engagement = engagement_model.predict(sample_student)[0]

print("Predicted engagement score:", round(predicted_engagement, 4))

Predicted engagement score: 0.3321


In [20]:
os.makedirs("saved_models", exist_ok=True)

joblib.dump(tfidf, "saved_models/tfidf_vectorizer.pkl")
joblib.dump(courses, "saved_models/courses.pkl")
joblib.dump(
    engagement_model,
    "saved_models/engagement_model.pkl"
)

print("Models and course data saved successfully!")

Models and course data saved successfully!


In [21]:
loaded_tfidf = joblib.load(
    "saved_models/tfidf_vectorizer.pkl"
)

loaded_courses = joblib.load(
    "saved_models/courses.pkl"
)

loaded_engagement_model = joblib.load(
    "saved_models/engagement_model.pkl"
)

print("Saved files loaded successfully!")
print("Number of available courses:", len(loaded_courses))

Saved files loaded successfully!
Number of available courses: 50


In [22]:
recommend_courses(
    topic="Python",
    difficulty="Beginner",
    learning_style="Visual",
    top_n=5
)

No courses found for topic: Python


""


In [23]:
print("Available topics:")
print(sorted(df["topic"].dropna().unique()))

Available topics:
['AI', 'Art', 'Data Science', 'History', 'Math', 'Music']


In [24]:
print(df["topic"].value_counts())

topic
History         224
Art             192
Data Science    183
AI              145
Music           137
Math            126
Name: count, dtype: int64


In [25]:
print(df[["course_id", "title", "topic", "difficulty", "content_type"]]
      .drop_duplicates("course_id")
      .head(20))

   course_id      title         topic    difficulty content_type
0       C015  Course 15  Data Science      Beginner         Text
1       C029  Course 29       History      Advanced         Quiz
3       C011  Course 11  Data Science      Advanced        Video
4       C032  Course 32           Art      Beginner         Text
5       C002   Course 2          Math  Intermediate         Quiz
6       C047  Course 47           Art  Intermediate         Quiz
7       C014  Course 14  Data Science      Beginner         Text
8       C020  Course 20           Art      Beginner         Quiz
9       C022  Course 22       History      Advanced         Quiz
11      C027  Course 27            AI      Beginner         Quiz
12      C003   Course 3            AI      Advanced         Text
13      C017  Course 17  Data Science      Beginner        Video
14      C044  Course 44         Music      Advanced        Video
15      C028  Course 28           Art  Intermediate         Quiz
17      C021  Course 21  

In [26]:

def create_course_title(row):
    topic = row["topic"]
    difficulty = row["difficulty"]
    content_type = row["content_type"]

    topic_names = {
        "AI": "Artificial Intelligence",
        "Art": "Art and Creativity",
        "Data Science": "Data Science",
        "History": "World History",
        "Math": "Mathematics",
        "Music": "Music"
    }

    topic_name = topic_names.get(topic, topic)

    if difficulty == "Beginner":
        level = "Introduction to"
    elif difficulty == "Intermediate":
        level = "Intermediate"
    else:
        level = "Advanced"

    if content_type == "Video":
        content = "Video Course"
    elif content_type == "Quiz":
        content = "Practice Quiz"
    else:
        content = "Learning Fundamentals"

    if level == "Introduction to":
        return f"{level} {topic_name} - {content}"

    return f"{level} {topic_name} - {content}"


df["title"] = df.apply(create_course_title, axis=1)

print(df[["course_id", "title", "topic",
          "difficulty", "content_type"]]
      .drop_duplicates("course_id")
      .head(20))


   course_id                                              title         topic  \
0       C015  Introduction to Data Science - Learning Fundam...  Data Science   
1       C029             Advanced World History - Practice Quiz       History   
3       C011               Advanced Data Science - Video Course  Data Science   
4       C032  Introduction to Art and Creativity - Learning ...           Art   
5       C002           Intermediate Mathematics - Practice Quiz          Math   
6       C047    Intermediate Art and Creativity - Practice Quiz           Art   
7       C014  Introduction to Data Science - Learning Fundam...  Data Science   
8       C020  Introduction to Art and Creativity - Practice ...           Art   
9       C022             Advanced World History - Practice Quiz       History   
11      C027  Introduction to Artificial Intelligence - Prac...            AI   
12      C003  Advanced Artificial Intelligence - Learning Fu...            AI   
13      C017        Introduc

In [27]:
recommend_courses("Data Science")

,Course ID,Course Title,Topic,Difficulty,Content Type,Learning Style,Similarity Score (%)
0,C006,Course 6,Data Science,Beginner,Text,Auditory,72.77
1,C015,Course 15,Data Science,Beginner,Text,Kinesthetic,56.21
2,C014,Course 14,Data Science,Beginner,Text,Kinesthetic,56.21
3,C043,Course 43,Data Science,Beginner,Text,Kinesthetic,56.21
4,C038,Course 38,Data Science,Beginner,Quiz,Visual,56.05


In [31]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create features for each course
df["features"] = (
    df["topic"].astype(str) + " " +
    df["difficulty"].astype(str) + " " +
    df["content_type"].astype(str) + " " +
    df["learning_style"].astype(str)
)

# Convert course features into numerical vectors
tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(df["features"])

# Calculate similarity between courses
similarity = cosine_similarity(feature_matrix)

print("Similarity matrix created successfully!")
print("Shape:", similarity.shape)


Similarity matrix created successfully!
Shape: (1007, 1007)


In [32]:
recommend_courses("Data Science")

,Course ID,Course Title,Topic,Difficulty,Content Type
0,C014,Introduction to Data Science - Learning Fundam...,Data Science,Beginner,Text
1,C006,Introduction to Data Science - Learning Fundam...,Data Science,Beginner,Text
2,C043,Introduction to Data Science - Learning Fundam...,Data Science,Beginner,Text
3,C015,Introduction to Data Science - Learning Fundam...,Data Science,Beginner,Text
4,C045,Introduction to Data Science - Practice Quiz,Data Science,Beginner,Quiz
